# People counter: unified run (benchmark + alerts + clips, per video)

Give it **one video** or a **whole folder**. For **each video**, one after another, it runs:

* **Part A, model benchmark**: every model in `MODELS` → FPS, latency (p50/p95/p99), real-time pass/fail, GPU/CPU/RAM use, counts, and an HTML report.
* **Part B, alerts + clips**: counts people frame by frame, records **every** time the limit is exceeded (when it was first exceeded, alert time, duration, peak) and cuts clips from **3 s before the violation to 3 s after the alert** (overlapping alerts merge into one clip).

…then shows that video's **report, results table, alert table and playable clips** before moving to the next video. A cross-video summary comes at the end.

**How to use**
1. `Runtime ▸ Change runtime type ▸ GPU`.
2. Put videos under `videos/` (file browser ▸ drag & drop, the optional Drive-unzip cell, or the upload prompt). Set `VIDEO_PATH` in step 4 to **a single file** (e.g. `videos/lobby.mp4`) or **a folder** (`videos`).
3. `Runtime ▸ Run all`.

In [ ]:
#@title 1. Check the GPU
import subprocess, torch
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "nvidia-smi not found")
GPU = torch.cuda.is_available()
print("CUDA available:", GPU, "|", torch.cuda.get_device_name(0) if GPU else "")
if not GPU:
    print("⚠️  No GPU found. Runtime ▸ Change runtime type ▸ pick a GPU, then run all again. (It will still run on CPU, slowly.)")

In [ ]:
#@title 2. Install dependencies
!pip -q install ultralytics lap nvidia-ml-py "opencv-python<5"

In [ ]:
#@title 3. Write the peoplecount package (generated from the repo, no need to edit)
FILES = {
"peoplecount/__init__.py": "__version__ = \"0.1.0\"\n",
"peoplecount/__main__.py": "from .cli import main\nmain()\n",
"peoplecount/alert.py": "\"\"\"Threshold alerting. Every *episode* of the count exceeding the limit is recorded.\n\nEpisode lifecycle\n  start_t   : first moment the (smoothed) count went above the threshold  <- \"time it exceeded the limit\"\n  ALERT     : raised once the count has stayed above the threshold for `hold_s` (debounce against blips)\n  CLEARED   : raised once the count has stayed at/below the limit for `clear_s`; the episode's end time is the\n              last moment it was still over, so short dips below the limit do not split one episode in two.\nA new episode (= a new ALERT) can start immediately after a CLEARED (`cooldown_s` defaults to 0, no suppression).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport time\nimport urllib.request\nfrom dataclasses import asdict, dataclass\nfrom typing import Callable, Optional\n\n\ndef timecode(t: float) -> str:\n    \"\"\"Stream time in seconds -> HH:MM:SS.mmm\"\"\"\n    t = max(float(t), 0.0)\n    h, rem = divmod(t, 3600)\n    m, s = divmod(rem, 60)\n    return f\"{int(h):02d}:{int(m):02d}:{s:06.3f}\"\n\n\n@dataclass\nclass AlertEvent:\n    kind: str                  # \"ALERT\" | \"CLEARED\"\n    episode: int               # 1, 2, 3 ... one per exceedance episode\n    t: float                   # stream time of this event (s)\n    timecode: str\n    count: int                 # count when the event fired\n    threshold: int\n    start_t: float             # when the limit was FIRST exceeded in this episode\n    start_timecode: str\n    peak_count: int            # highest count so far in the episode\n    peak_t: float\n    wall_time: float           # epoch seconds when the event was raised\n    wall_time_iso: str\n    end_t: Optional[float] = None          # CLEARED only: last moment still over the limit\n    end_timecode: Optional[str] = None\n    duration_s: Optional[float] = None     # CLEARED only: end_t - start_t\n    reason: str = \"\"                       # \"stream_end\" if the video ended mid-episode\n\n\nclass ConsoleSink:\n    def __call__(self, e: AlertEvent):\n        if e.kind == \"ALERT\":\n            print(f\"\\033[91m🚨 ALERT #{e.episode}\\033[0m {e.count} people (limit {e.threshold}) | exceeded at \"\n                  f\"{e.start_timecode} | alert raised {e.timecode}\", flush=True)\n        else:\n            print(f\"\\033[92m✅ CLEARED #{e.episode}\\033[0m over limit {e.start_timecode} -> {e.end_timecode} \"\n                  f\"({e.duration_s:.1f}s), peak {e.peak_count}\" + (f\" [{e.reason}]\" if e.reason else \"\"), flush=True)\n\n\nclass JsonlSink:\n    def __init__(self, path: str):\n        self.path = path\n\n    def __call__(self, e: AlertEvent):\n        with open(self.path, \"a\") as f:\n            f.write(json.dumps(asdict(e)) + \"\\n\")\n\n\nclass WebhookSink:\n    \"\"\"POSTs the event as JSON (Slack-compatible 'text' field included). Failures never stop the pipeline.\"\"\"\n\n    def __init__(self, url: str, timeout: float = 2.0):\n        self.url, self.timeout = url, timeout\n\n    def __call__(self, e: AlertEvent):\n        body = asdict(e)\n        body[\"text\"] = (f\"{e.kind} #{e.episode}: {e.count} people (limit {e.threshold}), \"\n                        f\"exceeded at {e.start_timecode}, peak {e.peak_count}\")\n        req = urllib.request.Request(self.url, json.dumps(body).encode(), {\"Content-Type\": \"application/json\"})\n        try:\n            urllib.request.urlopen(req, timeout=self.timeout).read()\n        except Exception as ex:  # noqa: BLE001\n            print(f\"[webhook] failed: {ex}\", flush=True)\n\n\nclass AlertManager:\n    def __init__(self, threshold: int, hold_s: float = 2.0, clear_s: float = 3.0, hysteresis: int = 0,\n                 cooldown_s: float = 0.0, sinks: Optional[list[Callable[[AlertEvent], None]]] = None):\n        self.threshold, self.hold_s, self.clear_s = threshold, hold_s, clear_s\n        self.hysteresis, self.cooldown_s = hysteresis, cooldown_s\n        self.sinks = sinks or []\n        self.active = False\n        self.events: list[AlertEvent] = []\n        self.episodes: list[dict] = []          # one dict per episode (see _start_episode); end fields filled on clear\n        self._cur: Optional[dict] = None\n        self._over_since: Optional[float] = None\n        self._under_since: Optional[float] = None\n        self._last_over: float = 0.0\n        self._cand_peak: tuple[int, float] = (0, 0.0)\n        self._last_alert: float = -1e18\n\n    # -- main entry: call once per processed frame -------------------------------------------------------------\n    def update(self, count: int, t: float) -> Optional[AlertEvent]:\n        ev = None\n        if count > self.threshold:\n            self._under_since = None\n            self._last_over = t\n            if self._over_since is None:\n                self._over_since, self._cand_peak = t, (count, t)\n            elif count > self._cand_peak[0]:\n                self._cand_peak = (count, t)\n            if self.active:\n                if count > self._cur[\"peak_count\"]:\n                    self._cur[\"peak_count\"], self._cur[\"peak_t\"] = count, t\n            elif t - self._over_since >= self.hold_s and t - self._last_alert >= self.cooldown_s:\n                ev = self._start_episode(t, count)\n        else:\n            self._over_since = None\n            if self.active and count <= self.threshold - self.hysteresis:\n                if self._under_since is None:\n                    self._under_since = t\n                if t - self._under_since >= self.clear_s:\n                    ev = self._end_episode(t, count)\n            else:\n                self._under_since = None\n        return ev\n\n    def finish(self, t: float, count: int = 0) -> Optional[AlertEvent]:\n        \"\"\"Call when the stream ends so an episode still in progress is closed and recorded.\"\"\"\n        if self.active:\n            return self._end_episode(t, count, reason=\"stream_end\")\n        return None\n\n    # -- internals -----------------------------------------------------------------------------------------------\n    def _start_episode(self, t, count):\n        self.active, self._last_alert = True, t\n        peak, peak_t = self._cand_peak\n        self._cur = {\"episode\": len(self.episodes) + 1, \"threshold\": self.threshold,\n                     \"start_t\": round(self._over_since, 3), \"start_timecode\": timecode(self._over_since),\n                     \"alert_t\": round(t, 3), \"alert_timecode\": timecode(t), \"peak_count\": peak, \"peak_t\": round(peak_t, 3),\n                     \"end_t\": None, \"end_timecode\": None, \"duration_s\": None, \"clip\": None,\n                     \"start_wall_time_iso\": time.strftime(\"%Y-%m-%dT%H:%M:%S%z\", time.localtime())}\n        self.episodes.append(self._cur)\n        return self._emit(\"ALERT\", t, count)\n\n    def _end_episode(self, t, count, reason=\"\"):\n        c = self._cur\n        end_t = max(self._last_over, c[\"start_t\"])\n        c.update(end_t=round(end_t, 3), end_timecode=timecode(end_t), duration_s=round(end_t - c[\"start_t\"], 3))\n        self.active, self._under_since = False, None\n        return self._emit(\"CLEARED\", t, count, reason)\n\n    def _emit(self, kind: str, t: float, count: int, reason: str = \"\") -> AlertEvent:\n        c, now = self._cur, time.time()\n        e = AlertEvent(kind, c[\"episode\"], round(t, 3), timecode(t), count, self.threshold, c[\"start_t\"],\n                       c[\"start_timecode\"], c[\"peak_count\"], c[\"peak_t\"], now,\n                       time.strftime(\"%Y-%m-%dT%H:%M:%S%z\", time.localtime(now)),\n                       c[\"end_t\"] if kind == \"CLEARED\" else None, c[\"end_timecode\"] if kind == \"CLEARED\" else None,\n                       c[\"duration_s\"] if kind == \"CLEARED\" else None, reason)\n        self.events.append(e)\n        for s in self.sinks:\n            s(e)\n        return e\n",
"peoplecount/benchmark.py": "\"\"\"Run N models over the same video; store per-model JSON (latency stats, FPS, resources, per-frame counts).\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import hardware\nfrom .alert import AlertManager\nfrom .counter import PeopleCounter\nfrom .models import Detector, DEFAULT_SET, resolve, safe_name\nfrom .video import VideoSource\n\n\ndef pct(a, q):\n    return float(np.percentile(a, q)) if len(a) else float(\"nan\")\n\n\ndef bench_one(name, video, args, hw, tag):\n    spec = resolve(name)\n    if args.threads and args.device == \"cpu\":\n        import torch\n        torch.set_num_threads(args.threads)\n        os.environ[\"OMP_NUM_THREADS\"] = str(args.threads)\n    t_load = time.perf_counter()\n    det = Detector(spec, args.device, args.conf, args.imgsz, threads=args.threads)\n    probe = VideoSource(video, max_frames=1)\n    first = next(iter(probe))[2]\n    probe.close()\n    det.warmup(first, args.warmup)\n    load_s = time.perf_counter() - t_load\n\n    src = VideoSource(video, target_fps=args.target_fps, max_frames=args.max_frames)\n    pc = PeopleCounter(det, fps=src.eff_fps, track=not args.no_track, tracker=args.tracker)\n    am = AlertManager(args.threshold, hold_s=args.hold, clear_s=args.clear, cooldown_s=args.cooldown)\n    sampler = hardware.ResourceSampler().start()\n    lat, det_t, trk_t, idxs, raw, tracked, smooth, flags = [], [], [], [], [], [], [], []\n    t_wall = time.perf_counter()\n    t_dec = 0.0\n    it = iter(src)\n    while True:\n        t0 = time.perf_counter()\n        try:\n            idx, t, frame = next(it)\n        except StopIteration:\n            break\n        t_dec += time.perf_counter() - t0\n        t1 = time.perf_counter()\n        r = pc.step(frame)\n        lat.append(time.perf_counter() - t1)\n        det_t.append(r.t_detect); trk_t.append(r.t_track)\n        am.update(r.smooth, t)\n        idxs.append(idx); raw.append(r.raw); tracked.append(r.tracked); smooth.append(r.smooth); flags.append(int(am.active))\n    wall = time.perf_counter() - t_wall\n    res = sampler.stop()\n    src.close()\n    n = len(lat)\n    ms = np.array(lat) * 1e3\n    out = {\n        \"model\": spec.name, \"backend\": spec.backend, \"weights\": spec.weights, \"imgsz\": det.imgsz, \"device\": det.device,\n        \"half\": det.half, \"tag\": tag, \"hardware\": hw, \"video\": os.path.basename(video), \"video_fps\": src.fps,\n        \"video_size\": src.size, \"target_fps\": args.target_fps, \"detect_fps\": src.eff_fps, \"frames\": n, \"conf\": args.conf, \"threshold\": args.threshold,\n        \"load_warmup_s\": round(load_s, 2),\n        \"latency_ms\": {\"mean\": ms.mean(), \"p50\": pct(ms, 50), \"p95\": pct(ms, 95), \"p99\": pct(ms, 99),\n                       \"max\": ms.max(), \"detect_mean\": np.mean(det_t) * 1e3, \"track_mean\": np.mean(trk_t) * 1e3,\n                       \"decode_mean\": t_dec / max(n, 1) * 1e3},\n        \"fps_inference\": 1000 / ms.mean(),               # model+tracker only\n        \"fps_end_to_end\": n / wall,                      # incl. video decode\n        \"fps_p95_worst\": 1000 / pct(ms, 95),              # sustained-ish worst case\n        \"realtime_factor\": (n / wall) / src.eff_fps,  # >=1 => keeps up with the detection rate\n        \"resources\": res,\n        \"alerts\": sum(e.kind == \"ALERT\" for e in am.events), \"alert_events\": [vars(e) for e in am.events],\n        \"series\": {\"frame_idx\": idxs, \"raw\": raw, \"tracked\": tracked, \"smooth\": smooth, \"alert_active\": flags},\n        \"unique_people\": len(pc.seen), \"max_count\": int(max(smooth)) if smooth else 0,\n    }\n    return out\n\n\ndef run(args):\n    hw = hardware.collect(not args.no_cloud_detect)\n    dev = Detector._pick_device(args.device)\n    args.device = dev\n    tag = args.tag or hardware.auto_tag(hw, dev)\n    outdir = Path(args.out)\n    outdir.mkdir(parents=True, exist_ok=True)\n    models = args.models or DEFAULT_SET\n    print(f\"hardware tag: {tag}\\n{json.dumps({k: v for k, v in hw.items()}, indent=1)}\", flush=True)\n    written, summary = [], []\n    bar = args.target_fps or 25.0   # real-time bar: the detection rate you asked for (25 FPS if you asked for every frame)\n    for video in args.video:\n        for m in models:\n            print(f\"\\n=== {m} on {tag} | {os.path.basename(video)}\", flush=True)\n            try:\n                r = bench_one(m, video, args, hw, tag)\n            except Exception as e:  # noqa: BLE001  one bad model must not kill the sweep\n                print(f\"  FAILED: {type(e).__name__}: {e}\", flush=True)\n                continue\n            f = outdir / f\"{tag}__{safe_name(m)}__{Path(video).stem}.json\"\n            f.write_text(json.dumps(r, indent=1, default=float))\n            written.append(f); summary.append(r)\n            print(f\"  {r['fps_end_to_end']:.1f} FPS e2e | p95 {r['latency_ms']['p95']:.1f} ms | \"\n                  f\"max count {r['max_count']} | alerts {r['alerts']} | rt-factor {r['realtime_factor']:.2f} -> {f}\", flush=True)\n\n    if summary:\n        print_summary(summary, bar)\n    if summary and not args.no_report:\n        from .report import build\n        stamp = time.strftime(\"%Y%m%d-%H%M%S\")\n        rep = Path(args.reports) / f\"report_{tag}_{stamp}.html\"\n        rep.parent.mkdir(parents=True, exist_ok=True)\n        build(outdir, rep, args.gt, bar, files=written)\n        print(f\"\\nReport for this run: {rep}  (+ {rep.with_suffix('.csv')})\")\n        print(\"All runs combined:   python -m peoplecount report\")\n\n\ndef print_summary(rs, target_fps):\n    \"\"\"Console table at the end of every run: one row per (video, model).\"\"\"\n    hdr = f\"{'video':22s} {'model':20s} {'E2E FPS':>8s} {'model FPS':>9s} {'p50 ms':>7s} {'p95 ms':>7s} {'p99 ms':>7s} \" \\\n          f\"{'RT x':>5s} {'avg/max':>8s} {'alerts':>6s} {'CPU%':>6s} {'RAM MB':>7s} {'GPU%':>5s} {'VRAM MB':>8s} {'RT?':>4s}\"\n    print(\"\\n\" + \"=\" * len(hdr) + f\"\\nRESULTS on {rs[0]['tag']}  (real-time = >= {target_fps:g} FPS and p95 <= {1000/target_fps:.0f} ms)\\n\" + hdr)\n    f = lambda v, fmt: \"-\" if v is None else format(v, fmt)  # noqa: E731\n    for r in sorted(rs, key=lambda r: (r[\"video\"], -r[\"fps_end_to_end\"])):\n        l, u = r[\"latency_ms\"], r[\"resources\"]\n        ok = r[\"fps_end_to_end\"] >= target_fps and l[\"p95\"] <= 1000 / target_fps\n        avg = sum(r[\"series\"][\"smooth\"]) / max(len(r[\"series\"][\"smooth\"]), 1)\n        print(f\"{r['video'][:22]:22s} {r['model'][:20]:20s} {r['fps_end_to_end']:8.1f} {r['fps_inference']:9.1f} {l['p50']:7.1f} \"\n              f\"{l['p95']:7.1f} {l['p99']:7.1f} {r['realtime_factor']:5.2f} {avg:4.1f}/{r['max_count']:<3d} {r['alerts']:6d} \"\n              f\"{f(u['cpu_pct_avg'], '6.0f'):>6s} {u['rss_peak_mb']:7d} {f(u['gpu_util_avg'], '5.0f'):>5s} \"\n              f\"{f(u['gpu_mem_peak_mb'], '8.0f'):>8s} {'yes' if ok else 'NO':>4s}\")\n    print(\"=\" * len(hdr))\n",
"peoplecount/cli.py": "import argparse\nimport json\n\nfrom .models import DEFAULT_SET, REGISTRY\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(prog=\"peoplecount\", description=__doc__)\n    sub = p.add_subparsers(dest=\"cmd\", required=True)\n\n    def common(s):\n        s.add_argument(\"--device\", default=\"auto\", help=\"auto|cpu|cuda:0|mps\")\n        s.add_argument(\"--conf\", type=float, default=0.3)\n        s.add_argument(\"--threads\", type=int, help=\"CPU threads (try 4; on hybrid P/E-core laptops fewer is often faster)\")\n        s.add_argument(\"--imgsz\", type=int, default=None, help=\"override model input size\")\n        s.add_argument(\"--threshold\", type=int, default=10, help=\"alert when people count exceeds this\")\n        s.add_argument(\"--hold\", type=float, default=2.0, help=\"seconds over threshold before ALERT\")\n        s.add_argument(\"--clear\", type=float, default=3.0, help=\"seconds under threshold before CLEARED\")\n        s.add_argument(\"--cooldown\", type=float, default=0.0, help=\"min seconds between alerts (0 = every episode is alerted)\")\n        s.add_argument(\"--target-fps\", type=float, default=None, help=\"run detection at this many frames per second \"\n                       \"(default: every frame). Frames are picked evenly in time; can't exceed the video's own FPS\")\n        s.add_argument(\"--tracker\", default=\"bytetrack\", choices=[\"bytetrack\", \"botsort\"])\n\n    s = sub.add_parser(\"list-models\"); s.set_defaults(f=\"list\")\n    s = sub.add_parser(\"hardware\"); s.set_defaults(f=\"hw\")\n\n    s = sub.add_parser(\"run\", help=\"real-time count + alert on a file/webcam/RTSP\")\n    s.add_argument(\"source\"); s.add_argument(\"--model\", default=\"yolo11n\"); common(s)\n    s.add_argument(\"--realtime\", action=\"store_true\", help=\"pace a file at its native FPS and drop frames when slow\")\n    s.add_argument(\"--show\", action=\"store_true\")\n    s.add_argument(\"--out-dir\", default=\"samples\", help=\"outputs go to <out-dir>/<video-name>/<time>_<uuid>_<model>_<video-name>.*\")\n    s.add_argument(\"--save\", help=\"override the annotated-mp4 path\")\n    s.add_argument(\"--no-clips\", action=\"store_true\", help=\"do not cut evidence clips\")\n    s.add_argument(\"--clip-pre\", type=float, default=3.0, help=\"seconds of footage kept BEFORE the violation started\")\n    s.add_argument(\"--clip-post\", type=float, default=3.0, help=\"seconds kept AFTER the alert was raised\")\n    s.add_argument(\"--merge-gap\", type=float, default=1.0, help=\"alerts whose clips are closer than this are merged into one clip\")\n    s.add_argument(\"--no-save\", action=\"store_true\", help=\"do not write a video\")\n    s.add_argument(\"--alert-log\", help=\"override the alert JSONL path\"); s.add_argument(\"--webhook\"); s.set_defaults(f=\"run\")\n\n    s = sub.add_parser(\"benchmark\", help=\"compare models on your video(s) on THIS machine\")\n    s.add_argument(\"video\", nargs=\"+\"); s.add_argument(\"--models\", nargs=\"*\", help=f\"default: {' '.join(DEFAULT_SET)}\")\n    common(s)\n    s.add_argument(\"--out\", default=\"results\"); s.add_argument(\"--tag\"); s.add_argument(\"--max-frames\", type=int)\n    s.add_argument(\"--warmup\", type=int, default=10)\n    s.add_argument(\"--no-track\", action=\"store_true\"); s.add_argument(\"--no-cloud-detect\", action=\"store_true\")\n    s.add_argument(\"--reports\", default=\"reports\", help=\"folder for the per-run HTML/CSV report\")\n    s.add_argument(\"--no-report\", action=\"store_true\"); s.add_argument(\"--gt\", help=\"ground-truth CSV (frame,count)\")\n    s.set_defaults(f=\"bench\")\n\n    s = sub.add_parser(\"report\", help=\"HTML/CSV comparison across everything in results/\")\n    s.add_argument(\"--results\", default=\"results\"); s.add_argument(\"--out\", default=\"report.html\")\n    s.add_argument(\"--gt\", help=\"CSV with columns frame,count (optional ground truth)\")\n    s.add_argument(\"--target-fps\", type=float, default=25.0, help=\"real-time bar: FPS a model must sustain to pass\"); s.set_defaults(f=\"report\")\n\n    s = sub.add_parser(\"export\", help=\"export to onnx/openvino/engine(TensorRT)/ncnn...\")\n    s.add_argument(\"model\"); s.add_argument(\"--format\", required=True); s.add_argument(\"--imgsz\", type=int, default=640)\n    s.add_argument(\"--half\", action=\"store_true\"); s.add_argument(\"--int8\", action=\"store_true\"); s.set_defaults(f=\"export\")\n\n    a = p.parse_args(argv)\n    if a.f == \"list\":\n        for m in REGISTRY.values():\n            print(f\"{m.name:22s} {m.backend:12s} {m.weights:42s} {m.note}\")\n    elif a.f == \"hw\":\n        from . import hardware\n        print(json.dumps(hardware.collect(), indent=1))\n    elif a.f == \"run\":\n        from .live import run; run(a)\n    elif a.f == \"bench\":\n        from .benchmark import run; run(a)\n    elif a.f == \"report\":\n        from .report import build; print(\"wrote\", build(a.results, a.out, a.gt, a.target_fps))\n    elif a.f == \"export\":\n        from .export import export; export(a.model, a.format, a.imgsz, a.half, a.int8)\n",
"peoplecount/clips.py": "\"\"\"Evidence clips: `pre_s` seconds before the violation started until `post_s` seconds after the alert was raised.\n\nOverlapping / back-to-back alerts are merged into ONE clip instead of producing many near-duplicates:\n  * annotated frames of the last few seconds are kept in a RAM ring buffer (JPEG-compressed, ~10-15 MB for 8 s of 1080p);\n  * a clip starts when an alert fires (frames are taken from the buffer, so the pre-roll is real footage);\n  * a later alert whose pre-roll window overlaps (or is within `merge_gap_s` of) the open clip just EXTENDS it;\n  * the clip is finalised only when no later alert could still merge into it (or the stream ends).\n\"\"\"\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom pathlib import Path\nfrom typing import Callable, Optional\n\nimport cv2\nimport numpy as np\n\n\ndef _fmt(t: float) -> str:\n    h, rem = divmod(int(t), 3600)\n    m, s = divmod(rem, 60)\n    return f\"{h:02d}h{m:02d}m{s:02d}s\"\n\n\nclass ClipRecorder:\n    def __init__(self, out_dir, pre_s: float = 3.0, post_s: float = 3.0, hold_s: float = 2.0,\n                 merge_gap_s: float = 1.0, jpeg_quality: int = 90,\n                 on_clip: Optional[Callable[[dict], None]] = None):\n        self.dir, self.pre, self.post, self.gap = Path(out_dir), pre_s, post_s, merge_gap_s\n        self.hold, self.q, self.on_clip = hold_s, jpeg_quality, on_clip\n        # an alert that could still merge into the open clip fires at most pre+hold+gap after the clip's end\n        self.settle = pre_s + hold_s + merge_gap_s\n        self.horizon = self.settle + 2.0\n        self.ring: deque = deque()          # (t, jpeg bytes)\n        self.clips: list[dict] = []         # finished clips: path, start_t, end_t, episodes\n        self._w: Optional[cv2.VideoWriter] = None\n        self._cur: Optional[dict] = None\n        self._end = 0.0                     # stream time up to which the open clip wants frames\n        self._last_t = -1.0                 # stream time of the last frame written to the open clip\n\n    # -- per frame (pass the ANNOTATED frame) ---------------------------------------------------------------------\n    def add_frame(self, t: float, frame: np.ndarray):\n        ok, buf = cv2.imencode(\".jpg\", frame, [cv2.IMWRITE_JPEG_QUALITY, self.q])\n        if ok:\n            self.ring.append((t, buf))\n        while self.ring and t - self.ring[0][0] > self.horizon:\n            self.ring.popleft()\n        if self._w is not None:\n            if t <= self._end:\n                self._w.write(frame)\n                self._last_t = t\n            elif t > self._end + self.settle:\n                self._finalize()\n\n    # -- when an ALERT fires (call AFTER add_frame for the same frame) ---------------------------------------------\n    def on_alert(self, start_t: float, alert_t: float, episode: int) -> Optional[str]:\n        want_start, new_end = start_t - self.pre, alert_t + self.post\n        if self._w is not None and want_start <= self._end + self.gap:\n            for t, buf in self.ring:                    # fill the gap between the old end and now\n                if t > self._last_t:\n                    self._w.write(cv2.imdecode(buf, cv2.IMREAD_COLOR))\n                    self._last_t = t\n            self._end = max(self._end, new_end)\n            self._cur[\"episodes\"].append(episode)\n            self._cur[\"merged\"] = True\n        else:\n            self._finalize()\n            self._open(want_start, new_end, episode)\n        return self._cur[\"path\"] if self._cur else None\n\n    def close(self):\n        self._finalize()\n\n    # -- internals -------------------------------------------------------------------------------------------------\n    def _open(self, want_start: float, new_end: float, episode: int):\n        frames = [(t, b) for t, b in self.ring if t >= want_start] or list(self.ring)[-1:]\n        if not frames:\n            return\n        ts = [t for t, _ in (self.ring if len(frames) < 3 else frames)]\n        dts = np.diff(ts)\n        fps = float(np.clip(1.0 / np.median(dts), 1.0, 120.0)) if len(dts) and np.median(dts) > 0 else 25.0\n        first = cv2.imdecode(frames[0][1], cv2.IMREAD_COLOR)\n        h, w = first.shape[:2]\n        self.dir.mkdir(parents=True, exist_ok=True)\n        path = str(self.dir / f\"clip{len(self.clips) + 1:03d}_{_fmt(frames[0][0])}_alert{episode}.mp4\")\n        self._w = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*\"mp4v\"), fps, (w, h))\n        for t, b in frames:\n            self._w.write(cv2.imdecode(b, cv2.IMREAD_COLOR))\n            self._last_t = t\n        self._end = new_end\n        self._cur = {\"path\": path, \"start_t\": round(frames[0][0], 3), \"end_t\": None, \"fps\": round(fps, 2),\n                     \"episodes\": [episode], \"merged\": False}\n\n    def _finalize(self):\n        if self._w is None:\n            return\n        self._w.release()\n        self._w = None\n        self._cur[\"end_t\"] = round(self._last_t, 3)\n        self._cur[\"duration_s\"] = round(self._cur[\"end_t\"] - self._cur[\"start_t\"], 3)\n        self.clips.append(self._cur)\n        if self.on_clip:\n            self.on_clip(self._cur)\n        self._cur = None\n",
"peoplecount/counter.py": "\"\"\"Detector + ByteTrack + count smoothing. One `step()` per frame.\"\"\"\nfrom __future__ import annotations\n\nimport time\nfrom collections import deque\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom .models import Detector\n\n\nclass _Dets:\n    \"\"\"Minimal Boxes-like view the Ultralytics trackers accept (conf, cls, xywh, boolean indexing).\"\"\"\n\n    def __init__(self, d: np.ndarray):\n        self.xyxy, self.conf = d[:, :4].astype(np.float32), d[:, 4].astype(np.float32)\n        self.cls = np.zeros(len(d), np.float32)\n        self.xywh = np.c_[(self.xyxy[:, :2] + self.xyxy[:, 2:]) / 2, self.xyxy[:, 2:] - self.xyxy[:, :2]]\n        self.data = d\n\n    def __len__(self):\n        return len(self.conf)\n\n    def __getitem__(self, i):\n        o = object.__new__(_Dets)\n        o.xyxy, o.conf, o.cls, o.xywh, o.data = (x[i] for x in (self.xyxy, self.conf, self.cls, self.xywh, self.data))\n        return o\n\n\ndef _make_tracker(kind: str, conf: float, buffer: int):\n    \"\"\"kind: bytetrack | botsort  (both ship with Ultralytics; no ReID model needed).\"\"\"\n    from pathlib import Path\n    from types import SimpleNamespace\n\n    import ultralytics\n    from ultralytics.trackers import BOTSORT, BYTETracker\n    from ultralytics.utils import YAML\n    cfg = YAML.load(Path(ultralytics.__file__).parent / \"cfg\" / \"trackers\" / f\"{kind}.yaml\")\n    cfg.update(track_buffer=buffer, track_high_thresh=conf, new_track_thresh=conf)\n    cfg[\"with_reid\"] = False\n    return {\"bytetrack\": BYTETracker, \"botsort\": BOTSORT}[kind](SimpleNamespace(**cfg))\n\n\n@dataclass\nclass FrameResult:\n    raw: int                      # detections above conf threshold\n    tracked: int                  # confirmed tracks visible this frame (de-duplicated, flicker-suppressed)\n    smooth: int                   # rolling median of `tracked` (used for alerting)\n    boxes: np.ndarray             # (M,4) tracked boxes\n    ids: np.ndarray               # (M,)\n    t_detect: float = 0.0\n    t_track: float = 0.0\n    unique_ids_seen: int = 0\n\n\nclass PeopleCounter:\n    def __init__(self, detector: Detector, fps: float = 30.0, smooth_window: int = 9,\n                 track: bool = True, tracker: str = \"bytetrack\", lost_buffer_s: float = 1.5):\n        self.det, self.track_enabled = detector, track\n        self.tracker = _make_tracker(tracker, detector.conf, max(5, int(lost_buffer_s * fps))) if track else None\n        self.hist: deque[int] = deque(maxlen=smooth_window)\n        self.seen: set[int] = set()\n\n    def step(self, frame: np.ndarray) -> FrameResult:\n        t0 = time.perf_counter()\n        d = self.det.detect(frame)\n        t1 = time.perf_counter()\n        if self.track_enabled:\n            rows = self.tracker.update(_Dets(d), frame)  # [x1,y1,x2,y2,id,score,cls,idx]\n            rows = np.asarray(rows).reshape(-1, 8)\n            boxes, ids = rows[:, :4], rows[:, 4].astype(int)\n        else:\n            boxes, ids = d[:, :4], np.arange(len(d))\n        t2 = time.perf_counter()\n        n = len(ids)\n        self.seen.update(int(i) for i in ids)\n        self.hist.append(n)\n        return FrameResult(len(d), n, int(np.median(self.hist)), boxes, ids, t1 - t0, t2 - t1, len(self.seen))\n",
"peoplecount/export.py": "\"\"\"Export Ultralytics models to edge runtimes. The result path can be passed straight to `benchmark --models`.\"\"\"\nfrom __future__ import annotations\n\n\ndef export(model: str, fmt: str, imgsz: int = 640, half: bool = False, int8: bool = False):\n    \"\"\"fmt: onnx | openvino (Intel CPU/iGPU/NPU) | engine (TensorRT, needs NVIDIA GPU) | ncnn | coreml | tflite\"\"\"\n    from ultralytics import YOLO\n    from .models import resolve\n    w = resolve(model).weights\n    out = YOLO(w).export(format=fmt, imgsz=imgsz, half=half, int8=int8, dynamic=False)\n    print(\"exported:\", out)\n    return out\n",
"peoplecount/hardware.py": "\"\"\"Collect hardware / cloud-instance metadata so results from different machines can be compared.\"\"\"\nfrom __future__ import annotations\n\nimport platform\nimport re\nimport urllib.request\n\n\ndef _http(url, headers=None, method=\"GET\", timeout=0.4):\n    try:\n        req = urllib.request.Request(url, headers=headers or {}, method=method)\n        return urllib.request.urlopen(req, timeout=timeout).read().decode()\n    except Exception:  # noqa: BLE001\n        return None\n\n\ndef cloud_info() -> dict:\n    \"\"\"Link-local metadata endpoints only (AWS IMDSv2 / GCP). Fast timeout; returns {} off-cloud.\"\"\"\n    tok = _http(\"http://169.254.169.254/latest/api/token\", {\"X-aws-ec2-metadata-token-ttl-seconds\": \"60\"}, \"PUT\")\n    if tok:\n        it = _http(\"http://169.254.169.254/latest/meta-data/instance-type\", {\"X-aws-ec2-metadata-token\": tok})\n        if it:\n            return {\"provider\": \"aws\", \"instance_type\": it}\n    mt = _http(\"http://metadata.google.internal/computeMetadata/v1/instance/machine-type\", {\"Metadata-Flavor\": \"Google\"})\n    if mt:\n        return {\"provider\": \"gcp\", \"instance_type\": mt.rsplit(\"/\", 1)[-1]}\n    return {}\n\n\ndef cpu_model() -> str:\n    try:\n        for line in open(\"/proc/cpuinfo\"):\n            if line.startswith(\"model name\"):\n                return line.split(\":\", 1)[1].strip()\n    except OSError:\n        pass\n    return platform.processor() or platform.machine()\n\n\ndef collect(detect_cloud: bool = True) -> dict:\n    import os\n\n    import psutil\n    info = {\"platform\": platform.platform(), \"cpu\": cpu_model(), \"cpu_cores_logical\": os.cpu_count(),\n            \"cpu_cores_physical\": psutil.cpu_count(logical=False), \"ram_gb\": round(psutil.virtual_memory().total / 2**30, 1),\n            \"python\": platform.python_version()}\n    try:\n        import torch\n        info[\"torch\"] = torch.__version__\n        info[\"cuda\"] = torch.version.cuda\n        if torch.cuda.is_available():\n            p = torch.cuda.get_device_properties(0)\n            info[\"gpu\"] = p.name\n            info[\"gpu_mem_gb\"] = round(p.total_memory / 2**30, 1)\n            info[\"gpu_count\"] = torch.cuda.device_count()\n    except ImportError:\n        pass\n    if detect_cloud:\n        info[\"cloud\"] = cloud_info()\n    return info\n\n\ndef auto_tag(info: dict, device: str) -> str:\n    c = info.get(\"cloud\") or {}\n    base = f\"{c['provider']}-{c['instance_type']}\" if c else None\n    if not base:\n        base = info.get(\"gpu\") if device.startswith(\"cuda\") and info.get(\"gpu\") else info.get(\"cpu\", \"cpu\")\n    tag = re.sub(r\"[^A-Za-z0-9.]+\", \"-\", base).strip(\"-\")[:48]\n    return f\"{tag}-{'gpu' if device.startswith('cuda') else device.split(':')[0]}\"\n\n\nclass ResourceSampler:\n    \"\"\"Background sampler: process CPU %, peak RSS, GPU util %, GPU mem MB, GPU power W (NVML if present).\"\"\"\n\n    def __init__(self, interval: float = 0.25):\n        import threading\n\n        import psutil\n        self.p, self.interval = psutil.Process(), interval\n        self.cpu, self.gpu_util, self.gpu_mem, self.gpu_pw, self.rss = [], [], [], [], 0\n        self._stop = threading.Event()\n        self._h = None\n        try:\n            import pynvml\n            pynvml.nvmlInit()\n            self._nv, self._h = pynvml, pynvml.nvmlDeviceGetHandleByIndex(0)\n        except Exception:  # noqa: BLE001\n            pass\n        self.t = threading.Thread(target=self._run, daemon=True)\n\n    def _run(self):\n        self.p.cpu_percent(None)\n        while not self._stop.wait(self.interval):\n            self.cpu.append(self.p.cpu_percent(None))\n            self.rss = max(self.rss, self.p.memory_info().rss)\n            if self._h:\n                try:\n                    u = self._nv.nvmlDeviceGetUtilizationRates(self._h)\n                    self.gpu_util.append(u.gpu)\n                    self.gpu_mem.append(self._nv.nvmlDeviceGetMemoryInfo(self._h).used / 2**20)\n                    self.gpu_pw.append(self._nv.nvmlDeviceGetPowerUsage(self._h) / 1000)\n                except Exception:  # noqa: BLE001\n                    pass\n\n    def start(self):\n        self.t.start()\n        return self\n\n    def stop(self) -> dict:\n        self._stop.set()\n        self.t.join(1)\n        avg = lambda x: round(sum(x) / len(x), 1) if x else None  # noqa: E731\n        return {\"cpu_pct_avg\": avg(self.cpu), \"cpu_pct_peak\": round(max(self.cpu), 1) if self.cpu else None,\n                \"rss_peak_mb\": round(self.rss / 2**20), \"gpu_util_avg\": avg(self.gpu_util),\n                \"gpu_mem_peak_mb\": round(max(self.gpu_mem)) if self.gpu_mem else None, \"gpu_power_avg_w\": avg(self.gpu_pw)}\n",
"peoplecount/live.py": "\"\"\"Real-time run: overlay + alerting on a file / webcam / RTSP stream.\"\"\"\nfrom __future__ import annotations\n\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom .alert import AlertManager, ConsoleSink, JsonlSink, WebhookSink, timecode\nfrom .clips import ClipRecorder\nfrom .counter import PeopleCounter\nfrom .models import Detector, resolve\nfrom .video import VideoSource\n\n\ndef output_paths(args, spec):\n    \"\"\"samples/<video-stem>/<YYYYmmdd-HHMMSS>_<uuid8>_<model>_<video-stem>{.mp4,.alerts.jsonl,.episodes.json,_clips/}\"\"\"\n    import re\n    import time\n    import uuid\n    from pathlib import Path\n    src = str(args.source)\n    stem = f\"camera{src}\" if src.isdigit() else (re.sub(r\"\\W+\", \"_\", src.split(\"://\", 1)[1]).strip(\"_\")[:40]\n                                              if \"://\" in src else Path(src).stem)\n    model = re.sub(r\"\\W+\", \"_\", Path(spec.name).name.rstrip(\"/\")).strip(\"_\")\n    run_id = f\"{time.strftime('%Y%m%d-%H%M%S')}_{uuid.uuid4().hex[:8]}_{model}_{stem}\"\n    d = Path(args.out_dir) / stem\n    d.mkdir(parents=True, exist_ok=True)\n    video = None if args.no_save else (args.save or str(d / f\"{run_id}.mp4\"))\n    alerts = args.alert_log or str(d / f\"{run_id}.alerts.jsonl\")\n    return video, alerts, str(d / f\"{run_id}.episodes.json\"), str(d / f\"{run_id}_clips\"), str(d / f\"{run_id}.counts.csv\")\n\n\ndef draw(frame, res, fps, threshold, alerting, model, t=None):\n    for (x1, y1, x2, y2), i in zip(res.boxes.astype(int), res.ids):\n        cv2.rectangle(frame, (x1, y1), (x2, y2), (60, 220, 60), 2)\n        cv2.putText(frame, f\"#{i}\", (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (60, 220, 60), 1)\n    col = (0, 0, 255) if alerting else (40, 40, 40)\n    cv2.rectangle(frame, (0, 0), (frame.shape[1], 34), col, -1)\n    cv2.putText(frame, f\"{model} | people: {res.smooth} (limit {threshold}) | {fps:4.1f} FPS\" +\n                (\"  !! OVER CAPACITY !!\" if alerting else \"\"), (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)\n    if t is not None:\n        cv2.putText(frame, timecode(t)[:-4], (8, frame.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)\n    return frame\n\n\ndef run(args):\n    spec = resolve(args.model)\n    args.save, args.alert_log, episodes_path, clips_dir, counts_path = output_paths(args, spec)\n    det = Detector(spec, args.device, args.conf, args.imgsz, threads=args.threads)\n    src = VideoSource(args.source, realtime=args.realtime, target_fps=args.target_fps)\n    det.warmup(next(iter(VideoSource(args.source, max_frames=1)))[2], 3)\n    pc = PeopleCounter(det, fps=src.eff_fps, tracker=args.tracker)\n    sinks = [ConsoleSink()] + ([JsonlSink(args.alert_log)] if args.alert_log else []) + \\\n            ([WebhookSink(args.webhook)] if args.webhook else [])\n    am = AlertManager(args.threshold, args.hold, args.clear, cooldown_s=args.cooldown, sinks=sinks)\n    clips = None if args.no_clips else ClipRecorder(\n        clips_dir, args.clip_pre, args.clip_post, hold_s=args.hold, merge_gap_s=args.merge_gap,\n        on_clip=lambda c: print(f\"🎞  clip saved: {c['path']} ({c['duration_s']:.1f}s, alerts {c['episodes']})\", flush=True))\n    writer, ema, n, t_start = None, 0.0, 0, time.perf_counter()\n    print(f\"video out: {args.save or '(disabled)'}\\nalert log: {args.alert_log}\\nepisodes : {episodes_path}\\n\"\n          f\"clips dir: {'(disabled)' if clips is None else clips_dir}\\ncounts   : {counts_path}\", flush=True)\n    print(f\"model={spec.name} device={det.device} source={args.source} {src.size} @ {src.fps:.1f}fps \"\n          f\"{'(live)' if src.live else ''}\", flush=True)\n    t_last, last_count, series = 0.0, 0, []\n    for idx, t, frame in src:\n        t0 = time.perf_counter()\n        res = pc.step(frame)\n        ev = am.update(res.smooth, t)\n        series.append((idx, round(t, 3), timecode(t), res.smooth, res.raw))\n        dt = time.perf_counter() - t0\n        ema = dt if not ema else 0.9 * ema + 0.1 * dt\n        n += 1\n        t_last, last_count = t, res.smooth\n        if args.show or args.save or clips:\n            out = draw(frame, res, 1 / ema, args.threshold, am.active, spec.name, t)\n            if clips:\n                clips.add_frame(t, out)\n                if ev and ev.kind == \"ALERT\":\n                    am.episodes[-1][\"clip\"] = clips.on_alert(ev.start_t, ev.t, ev.episode)\n            if args.save and writer is None:\n                writer = cv2.VideoWriter(args.save, cv2.VideoWriter_fourcc(*\"mp4v\"), src.eff_fps, src.size)\n            if writer:\n                writer.write(out)\n            if args.show:\n                cv2.imshow(\"people-counter\", out)\n                if cv2.waitKey(1) & 0xFF == ord(\"q\"):\n                    break\n        if n % 100 == 0:\n            print(f\"  frame {idx}: count={res.smooth} raw={res.raw} fps={1/ema:.1f} dropped={src.dropped}\", flush=True)\n    if writer:\n        writer.release()\n    am.finish(t_last, last_count)\n    if clips:\n        clips.close()\n        by_ep = {e: c[\"path\"] for c in clips.clips for e in c[\"episodes\"]}\n        for ep in am.episodes:\n            ep[\"clip\"] = by_ep.get(ep[\"episode\"], ep[\"clip\"])\n    import csv\n    import json\n    with open(counts_path, \"w\", newline=\"\") as f:           # people count at every processed frame\n        w = csv.writer(f)\n        w.writerow([\"frame\", \"t\", \"timecode\", \"people\", \"raw\"])\n        w.writerows(series)\n    smooth = [r[3] for r in series] or [0]\n    summary = {\"frames\": n, \"duration_s\": round(t_last, 3), \"video_fps\": src.fps, \"max_people\": int(max(smooth)),\n               \"avg_people\": round(sum(smooth) / len(smooth), 2), \"distinct_tracks\": len(pc.seen)}\n    with open(episodes_path, \"w\") as f:\n        json.dump({\"source\": str(args.source), \"model\": spec.name, \"threshold\": args.threshold, \"hold_s\": args.hold,\n                   \"clear_s\": args.clear, \"summary\": summary, \"counts_csv\": counts_path, \"episodes\": am.episodes,\n                   \"clips\": clips.clips if clips else []}, f, indent=1)\n    src.close()\n    wall = time.perf_counter() - t_start\n    print(f\"done: {n} frames in {wall:.1f}s ({n/wall:.1f} FPS), dropped={src.dropped}, \"\n          f\"alert episodes={len(am.episodes)}, clips={len(clips.clips) if clips else 0}, unique people tracked={len(pc.seen)}\")\n    for ep in am.episodes:\n        print(f\"  #{ep['episode']}: exceeded at {ep['start_timecode']} | alert {ep['alert_timecode']} | over limit for \"\n              f\"{ep['duration_s']:.1f}s | peak {ep['peak_count']} | clip: {ep['clip'] and ep['clip'].split('/')[-1]}\")\n",
"peoplecount/models.py": "\"\"\"Model registry: every entry is a *person detector*. Tracking is applied uniformly on top\n(ByteTrack via Ultralytics) so that the comparison between detectors is fair.\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass ModelSpec:\n    name: str\n    backend: str  # ultralytics | rtdetr | torchvision\n    weights: str  # weight file / torchvision constructor name\n    imgsz: int = 640\n    note: str = \"\"\n\n\nREGISTRY: dict[str, ModelSpec] = {s.name: s for s in [\n    ModelSpec(\"yolov8n\", \"ultralytics\", \"yolov8n.pt\", note=\"nano, edge-friendly\"),\n    ModelSpec(\"yolov8s\", \"ultralytics\", \"yolov8s.pt\"),\n    ModelSpec(\"yolov8m\", \"ultralytics\", \"yolov8m.pt\"),\n    ModelSpec(\"yolo11n\", \"ultralytics\", \"yolo11n.pt\", note=\"nano, edge-friendly\"),\n    ModelSpec(\"yolo11s\", \"ultralytics\", \"yolo11s.pt\"),\n    ModelSpec(\"yolo11m\", \"ultralytics\", \"yolo11m.pt\"),\n    ModelSpec(\"yolo11l\", \"ultralytics\", \"yolo11l.pt\", note=\"accuracy reference\"),\n    ModelSpec(\"yolov10n\", \"ultralytics\", \"yolov10n.pt\", note=\"NMS-free\"),\n    ModelSpec(\"yolov10s\", \"ultralytics\", \"yolov10s.pt\", note=\"NMS-free\"),\n    ModelSpec(\"rtdetr-l\", \"rtdetr\", \"rtdetr-l.pt\", note=\"transformer, GPU-oriented\"),\n    ModelSpec(\"ssdlite_mobilenet\", \"torchvision\", \"ssdlite320_mobilenet_v3_large\", 320, \"lowest-cost torchvision\"),\n    ModelSpec(\"fasterrcnn_mobilenet\", \"torchvision\", \"fasterrcnn_mobilenet_v3_large_320_fpn\", 320),\n    ModelSpec(\"retinanet_r50\", \"torchvision\", \"retinanet_resnet50_fpn_v2\", 800, \"heavy\"),\n    ModelSpec(\"fasterrcnn_r50\", \"torchvision\", \"fasterrcnn_resnet50_fpn_v2\", 800, \"heavy, accurate\"),\n]}\n\n# Sensible default shortlist for a first run.\nDEFAULT_SET = [\"yolov8n\", \"yolo11n\", \"yolo11s\", \"yolov10n\", \"ssdlite_mobilenet\", \"fasterrcnn_mobilenet\", \"rtdetr-l\"]\n\n\ndef resolve(name: str) -> ModelSpec:\n    \"\"\"Registry name, or an arbitrary Ultralytics-loadable path (.pt/.onnx/.engine/openvino dir).\"\"\"\n    if name in REGISTRY:\n        return REGISTRY[name]\n    return ModelSpec(name, \"rtdetr\" if \"rtdetr\" in name.lower() else \"ultralytics\", name,\n                     note=\"custom/exported\")\n\n\ndef safe_name(name: str) -> str:\n    return name.replace(\"/\", \"_\").replace(\" \", \"_\")\n\n\nclass Detector:\n    \"\"\"detect(frame_bgr) -> float32 array (N,5): x1,y1,x2,y2,conf  (persons only).\"\"\"\n\n    def __init__(self, spec: ModelSpec, device: str = \"auto\", conf: float = 0.3,\n                 imgsz: int | None = None, half: bool = True, threads: int | None = None):\n        self.spec, self.conf = spec, conf\n        self.imgsz = imgsz or spec.imgsz\n        self.device = self._pick_device(device)\n        self.half = half and self.device.startswith(\"cuda\")\n        if threads and self.device == \"cpu\":\n            import torch\n            torch.set_num_threads(threads)\n        self._load()\n\n    @staticmethod\n    def _pick_device(device: str) -> str:\n        if device != \"auto\":\n            return device\n        import torch\n        if torch.cuda.is_available():\n            return \"cuda:0\"\n        if getattr(torch.backends, \"mps\", None) and torch.backends.mps.is_available():\n            return \"mps\"\n        return \"cpu\"\n\n    def _load(self):\n        s = self.spec\n        if s.backend in (\"ultralytics\", \"rtdetr\"):\n            from ultralytics import RTDETR, YOLO\n            self.model = (RTDETR if s.backend == \"rtdetr\" else YOLO)(s.weights)\n            self._fp16 = self.half and s.weights.endswith(\".pt\")\n        elif s.backend == \"torchvision\":\n            import torch\n            import torchvision.models.detection as tvd\n            self.model = getattr(tvd, s.weights)(weights=\"DEFAULT\").to(self.device).eval()\n            self._torch = torch\n        else:\n            raise ValueError(s.backend)\n\n    def detect(self, frame: np.ndarray) -> np.ndarray:\n        b = self.spec.backend\n        if b in (\"ultralytics\", \"rtdetr\"):\n            kw = {\"half\": True} if self._fp16 else {}\n            r = self.model.predict(frame, imgsz=self.imgsz, conf=self.conf, classes=[0],\n                                   device=self.device, verbose=False, **kw)[0]\n            if r.boxes is None or len(r.boxes) == 0:\n                return np.zeros((0, 5), np.float32)\n            return np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]]).astype(np.float32)\n        if b == \"torchvision\":\n            torch = self._torch\n            import cv2\n            h, w = frame.shape[:2]\n            scale = self.imgsz / min(h, w)  # torchvision transform resizes again; pre-shrink to save H2D bandwidth\n            if scale < 1:\n                frame_in = cv2.resize(frame, (int(w * scale), int(h * scale)))\n            else:\n                frame_in, scale = frame, 1.0\n            t = torch.from_numpy(frame_in[:, :, ::-1].copy()).permute(2, 0, 1).float().div_(255).to(self.device)\n            with torch.inference_mode():\n                out = self.model([t])[0]\n            keep = (out[\"labels\"] == 1) & (out[\"scores\"] >= self.conf)  # COCO person = 1 in torchvision\n            boxes = out[\"boxes\"][keep].cpu().numpy() / scale\n            scores = out[\"scores\"][keep].cpu().numpy()[:, None]\n            return np.hstack([boxes, scores]).astype(np.float32) if len(scores) else np.zeros((0, 5), np.float32)\n        raise ValueError(b)\n\n    def warmup(self, frame: np.ndarray, n: int = 5):\n        for _ in range(n):\n            self.detect(frame)\n",
"peoplecount/report.py": "\"\"\"Merge result JSONs (from any number of machines) into one HTML + CSV comparison report.\"\"\"\nfrom __future__ import annotations\n\nimport base64\nimport csv\nimport html\nimport io\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nPALETTE = [\"#2a6fdb\", \"#e8743b\", \"#19a979\", \"#9b59d0\", \"#d1478c\", \"#7f8c8d\", \"#c9a227\", \"#16a2b8\", \"#8c564b\", \"#4c5fd5\"]\n\n\ndef load(results_dir, files=None):\n    paths = [Path(f) for f in files] if files else sorted(Path(results_dir).glob(\"*.json\"))\n    rs = [json.loads(p.read_text()) for p in paths]\n    return [r for r in rs if \"series\" in r]\n\n\ndef load_gt(path):\n    gt = {}\n    with open(path) as f:\n        for row in csv.DictReader(f):\n            gt[int(row[\"frame\"])] = int(row[\"count\"])\n    return gt\n\n\ndef _mae(series, ref):\n    common = [i for i in series if i in ref]\n    if not common:\n        return None, None\n    e = np.array([series[i] - ref[i] for i in common], float)\n    return float(np.abs(e).mean()), float((np.abs(e) <= 0).mean() * 100)  # MAE, exact-match %\n\n\ndef _png(fig):\n    import matplotlib.pyplot as plt\n    b = io.BytesIO()\n    fig.savefig(b, format=\"png\", dpi=110, bbox_inches=\"tight\")\n    plt.close(fig)\n    return '<img src=\"data:image/png;base64,%s\">' % base64.b64encode(b.getvalue()).decode()\n\n\ndef build(results_dir, out_html, gt_path=None, target_fps=25.0, files=None):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    rs = load(results_dir, files)\n    if not rs:\n        raise SystemExit(f\"no results in {results_dir}\")\n    gt = load_gt(gt_path) if gt_path else None\n    by_video = defaultdict(list)\n    for r in rs:\n        by_video[r[\"video\"]].append(r)\n\n    # reference series per video: GT if given, else per-frame median of every model's smoothed count\n    refs, ref_kind = {}, {}\n    for v, rr in by_video.items():\n        per_model = {}\n        for r in rr:\n            per_model.setdefault(r[\"model\"], dict(zip(r[\"series\"][\"frame_idx\"], r[\"series\"][\"smooth\"])))\n        if gt:\n            refs[v], ref_kind[v] = gt, \"ground truth\"\n        elif len(per_model) < 2:  # a consensus of one model is meaningless\n            refs[v], ref_kind[v] = {}, \"n/a (need >=2 models or --gt for an error metric)\"\n        else:\n            frames = set.intersection(*[set(s) for s in per_model.values()])\n            refs[v] = {f: float(np.median([s[f] for s in per_model.values()])) for f in frames}\n            ref_kind[v] = \"consensus (median of all models) — agreement, NOT true accuracy\"\n\n    tags = sorted({r[\"tag\"] for r in rs})\n    models = sorted({r[\"model\"] for r in rs}, key=lambda m: -np.mean([r[\"fps_end_to_end\"] for r in rs if r[\"model\"] == m]))\n    col = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(models)}\n\n    rows = []\n    for r in rs:\n        s = dict(zip(r[\"series\"][\"frame_idx\"], r[\"series\"][\"smooth\"]))\n        mae, exact = _mae(s, refs[r[\"video\"]])\n        res, lat = r[\"resources\"], r[\"latency_ms\"]\n        rows.append({\"video\": r[\"video\"], \"hardware\": r[\"tag\"], \"model\": r[\"model\"], \"device\": r[\"device\"],\n                     \"fps_e2e\": r[\"fps_end_to_end\"], \"fps_infer\": r[\"fps_inference\"], \"p50_ms\": lat[\"p50\"],\n                     \"p95_ms\": lat[\"p95\"], \"p99_ms\": lat[\"p99\"], \"detect_ms\": lat[\"detect_mean\"], \"track_ms\": lat[\"track_mean\"],\n                     \"rt_factor\": r[\"realtime_factor\"], \"mae\": mae, \"exact_pct\": exact, \"max_count\": r[\"max_count\"], \"avg_count\": float(np.mean(r[\"series\"][\"smooth\"])), \"unique_people\": r[\"unique_people\"],\n                     \"alerts\": r[\"alerts\"], \"cpu_pct\": res[\"cpu_pct_avg\"], \"rss_mb\": res[\"rss_peak_mb\"],\n                     \"gpu_util\": res[\"gpu_util_avg\"], \"gpu_mem_mb\": res[\"gpu_mem_peak_mb\"], \"gpu_w\": res[\"gpu_power_avg_w\"],\n                     \"meets_target\": r[\"fps_end_to_end\"] >= target_fps and lat[\"p95\"] <= 1000 / target_fps})\n    out = Path(out_html)\n    with open(out.with_suffix(\".csv\"), \"w\", newline=\"\") as f:\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\n        w.writeheader(); w.writerows(rows)\n\n    figs = []\n    # 1. FPS grouped by model x hardware\n    fig, ax = plt.subplots(figsize=(max(7, len(models) * 1.1), 4.2))\n    wdt = 0.8 / len(tags)\n    for j, t in enumerate(tags):\n        ys = [np.mean([x[\"fps_e2e\"] for x in rows if x[\"model\"] == m and x[\"hardware\"] == t] or [0]) for m in models]\n        ax.bar(np.arange(len(models)) + j * wdt, ys, wdt, label=t, color=PALETTE[j % len(PALETTE)])\n    ax.axhline(target_fps, color=\"k\", ls=\"--\", lw=1); ax.text(0, target_fps, f\" real-time target {target_fps:g} FPS\", va=\"bottom\", fontsize=8)\n    ax.set_xticks(np.arange(len(models)) + wdt * (len(tags) - 1) / 2, models, rotation=30, ha=\"right\")\n    ax.set_ylabel(\"end-to-end FPS (higher is better)\"); ax.legend(fontsize=7); ax.set_title(\"Throughput by model and hardware\")\n    figs.append((\"Throughput\", _png(fig)))\n    # 2. p95 latency\n    fig, ax = plt.subplots(figsize=(max(7, len(models) * 1.1), 4.2))\n    for j, t in enumerate(tags):\n        ys = [np.mean([x[\"p95_ms\"] for x in rows if x[\"model\"] == m and x[\"hardware\"] == t] or [0]) for m in models]\n        ax.bar(np.arange(len(models)) + j * wdt, ys, wdt, label=t, color=PALETTE[j % len(PALETTE)])\n    ax.axhline(1000 / target_fps, color=\"k\", ls=\"--\", lw=1)\n    ax.set_xticks(np.arange(len(models)) + wdt * (len(tags) - 1) / 2, models, rotation=30, ha=\"right\")\n    ax.set_ylabel(\"p95 per-frame latency, ms (lower is better)\"); ax.legend(fontsize=7); ax.set_title(\"Tail latency\")\n    figs.append((\"Latency\", _png(fig)))\n    # 3. speed vs agreement\n    if any(x[\"mae\"] is not None for x in rows):\n        fig, ax = plt.subplots(figsize=(7, 4.5))\n        for x in rows:\n            if x[\"mae\"] is None:\n                continue\n            ax.scatter(x[\"fps_e2e\"], x[\"mae\"], s=60, color=col[x[\"model\"]], marker=\"os^Dv<>\"[tags.index(x[\"hardware\"]) % 8])\n            ax.annotate(x[\"model\"], (x[\"fps_e2e\"], x[\"mae\"]), fontsize=7, xytext=(4, 3), textcoords=\"offset points\")\n        ax.set_xscale(\"log\"); ax.set_xlabel(\"end-to-end FPS (log)\"); ax.set_ylabel(\"MAE of count vs reference (lower is better)\")\n        ax.set_title(\"Speed vs counting error (marker shape = hardware)\"); ax.grid(alpha=.3)\n        figs.append((\"Speed vs error\", _png(fig)))\n    # 4. count over time per video (one hardware: the first)\n    for v, rr in by_video.items():\n        fig, ax = plt.subplots(figsize=(10, 4))\n        seen = set()\n        for r in sorted(rr, key=lambda r: r[\"model\"]):\n            if r[\"model\"] in seen:\n                continue\n            seen.add(r[\"model\"])\n            ax.plot(np.array(r[\"series\"][\"frame_idx\"]) / r[\"video_fps\"], r[\"series\"][\"smooth\"], label=r[\"model\"], color=col[r[\"model\"]], lw=1.3)\n        if gt:\n            ax.plot(np.array(sorted(gt)) / rr[0][\"video_fps\"], [gt[k] for k in sorted(gt)], \"k--\", label=\"ground truth\", lw=1.5)\n        ax.axhline(rr[0][\"threshold\"], color=\"red\", ls=\":\", lw=1.2, label=f\"alert threshold ({rr[0]['threshold']})\")\n        ax.set_xlabel(\"time (s)\"); ax.set_ylabel(\"people (smoothed)\"); ax.legend(fontsize=7, ncol=2); ax.set_title(f\"Count over time — {v}\")\n        figs.append((f\"Counts: {v}\", _png(fig)))\n\n    cols = [(\"video\", \"Video\"), (\"hardware\", \"Hardware\"), (\"model\", \"Model\"), (\"fps_e2e\", \"E2E FPS\"), (\"p50_ms\", \"p50 ms\"),\n            (\"p95_ms\", \"p95 ms\"), (\"fps_infer\", \"model FPS\"), (\"p99_ms\", \"p99 ms\"), (\"detect_ms\", \"detect ms\"), (\"track_ms\", \"track ms\"), (\"rt_factor\", \"RT factor\"),\n            (\"mae\", \"MAE\"), (\"exact_pct\", \"exact %\"), (\"avg_count\", \"avg cnt\"), (\"max_count\", \"max cnt\"), (\"unique_people\", \"unique IDs\"), (\"alerts\", \"alerts\"), (\"cpu_pct\", \"CPU %\"),\n            (\"rss_mb\", \"RSS MB\"), (\"gpu_util\", \"GPU %\"), (\"gpu_mem_mb\", \"GPU MB\"), (\"gpu_w\", \"GPU W\"), (\"meets_target\", f\"≥{target_fps:g}FPS RT\")]\n\n    def cell(k, v):\n        if v is None:\n            return \"–\"\n        if isinstance(v, bool):\n            return \"✅\" if v else \"❌\"\n        return f\"{v:.1f}\" if isinstance(v, float) else html.escape(str(v))\n\n    table = \"<table><tr>\" + \"\".join(f\"<th>{h}</th>\" for _, h in cols) + \"</tr>\"\n    for x in sorted(rows, key=lambda x: (x[\"video\"], x[\"hardware\"], -x[\"fps_e2e\"])):\n        table += \"<tr>\" + \"\".join(f\"<td>{cell(k, x[k])}</td>\" for k, _ in cols) + \"</tr>\"\n    table += \"</table>\"\n    hwrows = \"\".join(f\"<tr><td>{html.escape(t)}</td><td>{html.escape(json.dumps({k: v for k, v in next(r for r in rs if r['tag']==t)['hardware'].items() if k in ('cpu','cpu_cores_logical','ram_gb','gpu','gpu_mem_gb','cloud','torch','cuda')}))}</td></tr>\" for t in tags)\n    refnote = \"\".join(f\"<li><b>{html.escape(v)}</b>: error measured against {k}</li>\" for v, k in ref_kind.items())\n    page = f\"\"\"<!doctype html><meta charset=utf-8><title>People counting model comparison</title>\n<style>body{{font:14px system-ui;max-width:1300px;margin:2em auto;padding:0 1em}}table{{border-collapse:collapse;font-size:12px}}\ntd,th{{border:1px solid #ccc;padding:3px 7px;text-align:right}}td:nth-child(-n+3),th:nth-child(-n+3){{text-align:left}}th{{background:#f2f2f2}}\nimg{{max-width:100%}}.w{{overflow-x:auto}}</style>\n<h1>People counting — model × hardware comparison</h1>\n<p>Real-time target: <b>{target_fps:g} FPS</b> end-to-end and p95 latency ≤ {1000/target_fps:.0f} ms. RT factor ≥ 1 means the pipeline keeps up with the source frame rate.\nCounts are tracker-confirmed (ByteTrack) and rolling-median smoothed.</p><ul>{refnote}</ul>\n<h2>Hardware</h2><table>{hwrows}</table>\n<h2>Results</h2><div class=w>{table}</div>\n{''.join(f'<h2>{t}</h2>{i}' for t, i in figs)}\"\"\"\n    out.write_text(page)\n    return out\n",
"peoplecount/video.py": "\"\"\"Video sources.\n\ntarget_fps: the rate at which frames are handed to the detector. The video is still read at its own speed, but only frames\nsampled evenly in TIME to this rate are decoded fully and returned (None = every frame). It can't exceed the source's own FPS.\nFiles: sequential, or paced in real time with `realtime=True` (frames we're too slow to process are dropped, not queued).\nLive (RTSP/webcam): a threaded latest-frame reader so a slow model drops stale frames instead of building lag.\n\"\"\"\nfrom __future__ import annotations\n\nimport threading\nimport time\n\nimport cv2\n\n\ndef _open(src):\n    s = int(src) if isinstance(src, str) and src.isdigit() else src\n    cap = cv2.VideoCapture(s)\n    if not cap.isOpened():\n        raise RuntimeError(f\"cannot open video source: {src}\")\n    return cap\n\n\ndef is_live(src) -> bool:\n    s = str(src)\n    return s.isdigit() or s.startswith((\"rtsp://\", \"rtmp://\", \"http://\", \"https://\", \"udp://\"))\n\n\nclass VideoSource:\n    def __init__(self, src, realtime: bool = False, target_fps: float | None = None, max_frames: int | None = None):\n        self.src, self.live = src, is_live(src)\n        self.cap = _open(src)\n        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0\n        if self.fps > 240 or self.fps < 1:\n            self.fps = 30.0\n        if target_fps is not None and target_fps <= 0:\n            raise ValueError(\"target_fps must be > 0\")\n        self.target_fps = float(target_fps) if target_fps else None\n        # the rate frames are actually handed to the detector (never above what the source provides)\n        self.eff_fps = min(self.target_fps, self.fps) if self.target_fps else self.fps\n        self.total = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT)) if not self.live else -1\n        self.size = (int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))\n        self.realtime, self.max_frames = realtime, max_frames\n        self.dropped = 0          # frames dropped because the detector was too slow (not counting frames skipped by target_fps)\n        self._latest, self._lock, self._stop = None, threading.Lock(), False\n        if self.live:\n            threading.Thread(target=self._reader, daemon=True).start()\n\n    def _reader(self):\n        n = 0\n        while not self._stop:\n            ok, f = self.cap.read()\n            if not ok:\n                time.sleep(0.5)\n                continue\n            n += 1\n            with self._lock:\n                self._latest = (n, f)\n\n    def __iter__(self):\n        \"\"\"yields (frame_index, stream_time_s, frame)\"\"\"\n        return self._iter_live() if self.live else self._iter_file()\n\n    def _iter_live(self):\n        last, t0, yielded, last_yield = 0, time.perf_counter(), 0, -1e18\n        min_dt = 1.0 / self.target_fps if self.target_fps else 0.0\n        while not self._stop:\n            now = time.perf_counter()\n            if min_dt and now - last_yield < min_dt:\n                time.sleep(min(0.002, min_dt - (now - last_yield)))\n                continue\n            with self._lock:\n                item = self._latest\n            if item is None or item[0] == last:\n                time.sleep(0.002)\n                continue\n            if not min_dt:\n                self.dropped += item[0] - last - 1 if last else 0\n            last, last_yield = item[0], now\n            yield last, now - t0, item[1]\n            yielded += 1\n            if self.max_frames and yielded >= self.max_frames:\n                return\n\n    def _iter_file(self):\n        cap, period = self.cap, 1.0 / self.eff_fps\n        idx, yielded, next_t, t0 = -1, 0, 0.0, time.perf_counter()\n        while True:\n            if not cap.grab():                       # grab() decodes without the costly copy/convert of read()\n                return\n            idx += 1\n            t = idx / self.fps\n            if t + 1e-9 < next_t:                    # not one of the evenly spaced detection slots\n                continue\n            next_t += period\n            if self.realtime:\n                lag = (time.perf_counter() - t0) - t\n                if lag > period:                     # more than one slot behind real time: drop it instead of queueing\n                    self.dropped += 1\n                    continue\n                if lag < 0:                          # ahead of real time: wait for this frame's moment\n                    time.sleep(-lag)\n            ok, frame = cap.retrieve()\n            if not ok:\n                return\n            yield idx, t, frame\n            yielded += 1\n            if self.max_frames and yielded >= self.max_frames:\n                return\n\n    def close(self):\n        self._stop = True\n        self.cap.release()\n"
}

import pathlib
for name, src in FILES.items():
    p = pathlib.Path(name); p.parent.mkdir(exist_ok=True); p.write_text(src)
print("wrote", len(FILES), "files to", pathlib.Path("peoplecount").resolve())

In [ ]:
#@title 4. Settings
import os, time
T0 = time.time()

#@markdown ### What to run
VIDEO_PATH = "videos"          #@param {type:"string"}
NAME_FILTER = ""               #@param {type:"string"}
#@markdown `VIDEO_PATH` = one video file **or** a folder (searched recursively). `NAME_FILTER` = only videos whose file name contains this text.
RUN_BENCHMARK = True           #@param {type:"boolean"}
RUN_ALERTS_AND_CLIPS = True    #@param {type:"boolean"}

#@markdown ### Part A: benchmark
MODELS = "yolov8n,yolo11n,yolo11s,yolo11m,yolo11l,rtdetr-l,ssdlite_mobilenet,fasterrcnn_mobilenet"  #@param {type:"string"}
MAX_FRAMES = 0                 #@param {type:"integer"}

#@markdown ### Part B: alerts + clips
ALERT_MODEL = "yolo11s"        #@param {type:"string"}
THRESHOLD = 10                 #@param {type:"integer"}
HOLD_S = 2.0                   #@param {type:"number"}
CLEAR_S = 3.0                  #@param {type:"number"}
CLIP_PRE_S = 3.0               #@param {type:"number"}
CLIP_POST_S = 3.0              #@param {type:"number"}
SAVE_FULL_ANNOTATED_VIDEO = True  #@param {type:"boolean"}

#@markdown ### Detection (both parts)
CONF = 0.3                     #@param {type:"number"}
IMGSZ = 0                      #@param {type:"integer"}
TARGET_FPS = 25               #@param {type:"number"}
#@markdown **TARGET_FPS** = how many frames per second to run detection on (0 = every frame). A 30 fps video with 5 here is analysed 5 times per second, and played in real time. It also sets the real-time bar in Part A (25 when 0).
TRACKER = "bytetrack"          #@param ["bytetrack", "botsort"]

#@markdown ### Display
SHOW_FULL_REPORT = True        #@param {type:"boolean"}
MAX_PREVIEWS_PER_VIDEO = 4     #@param {type:"integer"}

DEVICE = "cuda:0" if GPU else "cpu"
MODEL_LIST = [m.strip() for m in MODELS.split(",") if m.strip()]
print("device:", DEVICE, "| models:", MODEL_LIST, "| alert model:", ALERT_MODEL, "| alert when people >", THRESHOLD)

In [ ]:
#@title (optional) Load videos from a Google Drive zip
#@markdown Optional: unzip videos from a zip in your Google Drive (e.g. `MyDrive/videos.zip`). Leave empty to skip.
DRIVE_ZIP = ""  #@param {type:"string"}
if DRIVE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("videos", exist_ok=True)
    !unzip -qo "/content/drive/MyDrive/{DRIVE_ZIP}" -d videos
    print("unzipped into videos/")

In [ ]:
#@title 5. Find the video(s)
import glob, shutil, cv2
EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm", ".ogv", ".m4v", ".mpg", ".mpeg", ".wmv")

def find_videos():
    if os.path.isfile(VIDEO_PATH):
        found = [VIDEO_PATH]
    else:
        found = sorted(p for p in glob.glob(os.path.join(VIDEO_PATH, "**", "*"), recursive=True) if p.lower().endswith(EXTS))
    return [p for p in found if NAME_FILTER.lower() in os.path.basename(p).lower()]

VIDEOS = find_videos()
if not VIDEOS and not os.path.isfile(VIDEO_PATH):
    try:
        from google.colab import files
        os.makedirs(VIDEO_PATH, exist_ok=True)
        print(f"No videos found in {VIDEO_PATH}/. Choose files to upload (or drag them into the file browser and re-run this cell):")
        for name in files.upload():
            shutil.move(name, os.path.join(VIDEO_PATH, name))
        VIDEOS = find_videos()
    except ImportError:
        pass
assert VIDEOS, f"No videos found at {os.path.abspath(VIDEO_PATH)} (filter: {NAME_FILTER!r}). Put videos there or fix VIDEO_PATH."

print(f"{len(VIDEOS)} video(s) will be processed, one at a time:")
for v in VIDEOS:
    c = cv2.VideoCapture(v); fps = c.get(5) or 0
    print(f"  {v:60s} {int(c.get(3))}x{int(c.get(4))}  {fps:5.1f} fps  {c.get(7)/fps if fps else 0:6.1f}s")

In [ ]:
#@title Helpers
import sys, json, html as _html, traceback
import pandas as pd
from pathlib import Path
from IPython.display import HTML, Markdown, Video, display

def sh(args):
    """Run a command and stream its output live into the cell."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "it/s" in line or "━" in line or "Downloading" in line: continue   # skip progress-bar noise
        print(line, end="")
    return p.wait()

COMMON = ["--device", DEVICE, "--conf", str(CONF), "--threshold", str(THRESHOLD), "--hold", str(HOLD_S),
          "--clear", str(CLEAR_S), *(["--target-fps", str(TARGET_FPS)] if TARGET_FPS else []), "--tracker", TRACKER]
if IMGSZ: COMMON += ["--imgsz", str(IMGSZ)]

BENCH_COLS = ["model", "fps_e2e", "fps_infer", "p50_ms", "p95_ms", "p99_ms", "detect_ms", "track_ms", "rt_factor", "mae",
              "avg_count", "max_count", "unique_people", "alerts", "gpu_util", "gpu_mem_mb", "cpu_pct", "rss_mb", "meets_target"]

def bench_video(v):
    """Part A for one video -> DataFrame (one row per model) and report path."""
    t = time.time()
    cmd = [sys.executable, "-m", "peoplecount", "benchmark", v, "--models", *MODEL_LIST, *COMMON,
           "--out", "results", "--reports", "reports"]
    if MAX_FRAMES: cmd += ["--max-frames", str(MAX_FRAMES)]
    sh(cmd)
    reps = [r for r in sorted(glob.glob("reports/report_*.csv"), key=os.path.getmtime) if os.path.getmtime(r) >= t]
    if not reps:
        return None, None
    return pd.read_csv(reps[-1]), reps[-1].replace(".csv", ".html")

def alerts_video(v):
    """Part B for one video -> (episodes DataFrame, list of clip paths)."""
    t = time.time()
    cmd = [sys.executable, "-m", "peoplecount", "run", v, "--model", ALERT_MODEL, *COMMON,
           "--clip-pre", str(CLIP_PRE_S), "--clip-post", str(CLIP_POST_S), "--out-dir", "samples"]
    if not SAVE_FULL_ANNOTATED_VIDEO: cmd.append("--no-save")
    sh(cmd)
    stem = Path(v).stem
    rows, clips = [], []
    for f in sorted(glob.glob(f"samples/{stem}/*.episodes.json"), key=os.path.getmtime):
        if os.path.getmtime(f) < t: continue
        d = json.load(open(f))
        for ep in d["episodes"]:
            rows.append({"video": os.path.basename(d["source"]), "model": d["model"], "alert #": ep["episode"],
                         "exceeded at": ep["start_timecode"], "alert raised": ep["alert_timecode"],
                         "over limit (s)": ep["duration_s"], "peak": ep["peak_count"], "limit": ep["threshold"],
                         "clip": os.path.basename(ep["clip"]) if ep["clip"] else None, "_clip_path": ep["clip"]})
        clips += [c["path"] for c in d.get("clips", [])]
    return pd.DataFrame(rows), clips

def preview_clips(clip_paths, episodes_df):
    for p in clip_paths[:MAX_PREVIEWS_PER_VIDEO]:
        out = p.replace(".mp4", "_preview.mp4")      # browsers can't play the raw mp4v clips -> re-encode to H.264
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", p, "-vf", "scale=640:-2", "-c:v", "libx264",
                            "-crf", "28", "-pix_fmt", "yuv420p", out])
        if r.returncode == 0:
            eps = episodes_df.loc[episodes_df["_clip_path"] == p, "alert #"].tolist() if len(episodes_df) else []
            display(Markdown(f"`{os.path.basename(p)}` covers alert(s) {eps}"))
            display(Video(out, embed=True, width=520))

In [ ]:
#@title 6. Run every video: Part A + Part B, with results shown per video
BENCH_ALL, ALERTS_ALL, OVERVIEW = [], [], []

for k, v in enumerate(VIDEOS, 1):
    c = cv2.VideoCapture(v); fps = c.get(5) or 0
    display(Markdown(f"---\n# 🎬 Video {k}/{len(VIDEOS)}: `{os.path.basename(v)}`\n"
                     f"{int(c.get(3))}×{int(c.get(4))} · {fps:.1f} fps · {c.get(7)/fps if fps else 0:.1f} s"))
    row = {"video": os.path.basename(v)}
    try:
        # ---------------- Part A ----------------
        if RUN_BENCHMARK:
            display(Markdown("## Part A: model benchmark"))
            df, rep = bench_video(v)
            if df is not None:
                cols = [x for x in BENCH_COLS if x in df]
                display(df[cols].round(1).sort_values("fps_e2e", ascending=False).reset_index(drop=True))
                best = df.sort_values("fps_e2e", ascending=False).iloc[0]
                ok = df[df["meets_target"] == True]["model"].tolist()
                display(Markdown(f"**Fastest:** `{best['model']}` at {best['fps_e2e']:.1f} FPS · "
                                 f"**real-time (≥{(TARGET_FPS or 25):g} FPS):** {', '.join(ok) if ok else 'none'}"))
                row.update(fastest=best["model"], fastest_fps=round(best["fps_e2e"], 1))
                BENCH_ALL.append(df)
                if SHOW_FULL_REPORT:
                    display(Markdown(f"Full report with charts (`{rep}`):"))
                    display(HTML(f'<iframe srcdoc="{_html.escape(open(rep).read())}" width="100%" height="1000" '
                                 f'style="border:1px solid #ccc"></iframe>'))
        # ---------------- Part B ----------------
        if RUN_ALERTS_AND_CLIPS:
            display(Markdown(f"## Part B: alerts + clips (model `{ALERT_MODEL}`, limit {THRESHOLD})"))
            eps, clips = alerts_video(v)
            if len(eps):
                display(eps.drop(columns="_clip_path"))
                display(Markdown(f"**{len(eps)} violation(s)** · peak {int(eps['peak'].max())} people · "
                                 f"over the limit for {eps['over limit (s)'].sum():.1f} s in total · {len(clips)} clip(s)"))
                preview_clips(clips, eps)
                ALERTS_ALL.append(eps)
                row.update(violations=len(eps), peak=int(eps["peak"].max()), over_limit_s=round(eps["over limit (s)"].sum(), 1),
                           clips=len(clips))
            else:
                display(Markdown(f"✅ No violations: the count never stayed above {THRESHOLD} for {HOLD_S:g} s."))
                row.update(violations=0, peak=0, over_limit_s=0.0, clips=0)
    except Exception:
        display(Markdown("⚠️ **This video failed** (the rest will continue):"))
        traceback.print_exc()
        row["error"] = True
    OVERVIEW.append(row)

In [ ]:
#@title 7. Cross-video summary
display(Markdown("# 📊 Summary across all videos"))
if OVERVIEW:
    display(pd.DataFrame(OVERVIEW))
if BENCH_ALL:
    allb = pd.concat(BENCH_ALL)
    display(Markdown("**End-to-end FPS by model and video**"))
    display(allb.pivot_table(index="model", columns="video", values="fps_e2e").round(1))
    display(Markdown("**Which models were real-time (≥ target) on how many videos**"))
    display(allb.groupby("model").agg(videos=("video", "nunique"), mean_fps=("fps_e2e", "mean"),
                                      p95_ms=("p95_ms", "mean"), realtime_on=("meets_target", "sum")).round(1)
            .sort_values("mean_fps", ascending=False))
if ALERTS_ALL:
    display(Markdown("**All violations**"))
    display(pd.concat(ALERTS_ALL).drop(columns="_clip_path").reset_index(drop=True))

In [ ]:
#@title 8. Download everything (annotated videos, alert logs, clips, reports)
!zip -rq people_counter_outputs.zip samples results reports 2>/dev/null; ls -lh people_counter_outputs.zip
try:
    from google.colab import files
    files.download("people_counter_outputs.zip")
except ImportError:
    pass